# E11-01 · Mirror RedHat Qwen3.8-27B INT4 + z-lab DFlash2 → Kaggle Models (run this ON Kaggle)

**Notebook settings: Accelerator = None (CPU), Internet = ON.** No GPU quota is used.

### Before running: attach two Kaggle Secrets, start the run from the editor
Add-ons → Secrets → attach `KAGGLE_USERNAME` and `KAGGLE_KEY`, then **Save Version → Save & Run All** from the notebook
editor (a run started through the Kaggle API cannot reach the secrets service - proven in the ARC-AGI-3 project).

### How it works (no disk) - same code as the ARC-AGI-3 uploader that mirrored a 132.7 GB model
Every file streams from Hugging Face straight into Kaggle's resumable upload URL in 256 MiB chunks (RAM only).
Each file is hashed while it streams and compared with the pinned Hugging Face tree (LFS sha256 / git blob sha1); a model
instance is created **only if all its files match**. A dropped connection resumes from the byte the server confirmed.

### What gets published (both Apache-2.0; **private** models, `PUBLIC = False`. Other accounts can attach them only after
they are shared with them, or once `PUBLIC` is flipped)
| Hugging Face | revision | files | Kaggle path afterwards |
|---|---|---|---|
| `RedHatAI/Qwen3.8-27B-INT4` | `91bd022d5b49` | 11 (19.47 GB: model.safetensors 18.6 GB + model_mtp.safetensors 0.85 GB) | `/kaggle/input/models/<user>/qwen3-8-27b-int4-redhat/pytorch/w4a16-g128/1` |
| `z-lab/Qwen3.8-27B-DFlash2` | `50307d4c4cde` | 3 (3.85 GB BF16 drafter) | `/kaggle/input/models/<user>/qwen3-8-27b-dflash2/pytorch/bf16/1` |

`.gitattributes` and the DFlash2 README figure are not uploaded. The run is idempotent: an instance that already exists
is only listed.


In [ ]:
import hashlib
import json
import os
import queue
import shutil
import subprocess
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import requests

PUBLIC = False                      # user choice 2026-10-15: private Kaggle models (both checkpoints are Apache-2.0)
LICENSE_NAME = "Apache 2.0"
KAGGLE_FRAMEWORK = "pyTorch"        # appears as "pytorch" in the /kaggle/input path
HF_URL = "https://huggingface.co/{repo}/resolve/{rev}/{name}"
CHUNK = 256 << 20          # bytes per upload request (multiple of 256 KiB, as the resumable protocol requires)
QUANTUM = 256 << 10
READ = 8 << 20             # Hugging Face read size
WORKERS = 3
NOT_UPLOADED = {".gitattributes", "assets/dflash2-figure.png"}

# job -> pinned Hugging Face revision and every file: name -> [bytes, hash kind, hash]
# ("sha256" = LFS oid, "gitsha1" = git blob id of a non-LFS file)
JOBS = {
 "int4": {
  "hf_repo": "RedHatAI/Qwen3.8-27B-INT4",
  "slug": "qwen3-8-27b-int4-redhat",
  "title": "Qwen3.8 27B INT4 W4A16 (RedHat)",
  "instance": "w4a16-g128",
  "subtitle": "RedHat W4A16 g128 (compressed-tensors) + BF16 MTP head, for offline SGLang/vLLM",
  "revision": "91bd022d5b49442a868bc35008f6c21e1860edfa",
  "files": {
   ".gitattributes": [
    1570,
    "gitsha1",
    "52373fe24473b1aa44333d318f578ae6bf04b49b"
   ],
   "README.md": [
    6452,
    "gitsha1",
    "7799f5fba1417edaf5ad3490d1e6842c44b9b0e3"
   ],
   "chat_template.jinja": [
    8952,
    "gitsha1",
    "c0c686f9c38d70d179fb7b5f5aa7530bc913dda3"
   ],
   "config.json": [
    20952,
    "gitsha1",
    "c16a4740ae6ad1f4d335f679ceb2a93f33c7256c"
   ],
   "generation_config.json": [
    214,
    "gitsha1",
    "1f489fac1c66a575e119c65483a27147f0aab231"
   ],
   "model.safetensors": [
    18603387656,
    "sha256",
    "fd5c287e3a03f30b0e53fd5fe66ec90ab5fdd94807f7f8df699cca3ae9534910"
   ],
   "model.safetensors.index.json": [
    170752,
    "gitsha1",
    "b4a12b678f3099fa2bfa26c3745ff25482806f7d"
   ],
   "model_mtp.safetensors": [
    849400392,
    "sha256",
    "1d8268aa85ace093a561e3e7b63b9d390dac1cd55a90cd55b5ec509c3c9da9fe"
   ],
   "processor_config.json": [
    1220,
    "gitsha1",
    "43c4343ec493f200e452f87a7c6649ecdfcce1ac"
   ],
   "recipe.yaml": [
    1449,
    "gitsha1",
    "2cb819ba547c5a90f571976987a54faa2aae7e41"
   ],
   "tokenizer.json": [
    19989325,
    "sha256",
    "06b9509352d2af50381ab2247e083b80d32d5c0aba91c272ca9ff729b6a0e523"
   ],
   "tokenizer_config.json": [
    1166,
    "gitsha1",
    "b4acebe045132228e18a4e2ca7f75ce137d1fbf7"
   ]
  },
  "total": 19472990100
 },
 "dflash2": {
  "hf_repo": "z-lab/Qwen3.8-27B-DFlash2",
  "slug": "qwen3-8-27b-dflash2",
  "title": "Qwen3.8 27B DFlash2 drafter (z-lab)",
  "instance": "bf16",
  "subtitle": "DFlash2 BF16 speculative drafter for Qwen3.8-27B (SGLang --speculative-algorithm DFLASH)",
  "revision": "50307d4c4cde6860d4eee73e2547cd786fe8e8a4",
  "files": {
   ".gitattributes": [
    1581,
    "gitsha1",
    "50bbb90c56f7da6c41ef378fa235a891daf5a238"
   ],
   "README.md": [
    5397,
    "gitsha1",
    "1ee0eb2b2f81767f40521b61245211e5400b81bc"
   ],
   "assets/dflash2-figure.png": [
    286889,
    "sha256",
    "6d8dcc9a9472bddb644c881fe83050b5343d09bf5ffb0912e5598e55fe671e99"
   ],
   "config.json": [
    1239,
    "gitsha1",
    "79279cc5665bced6f3cdaa2095a2ffe819497b2e"
   ],
   "model.safetensors": [
    3848817896,
    "sha256",
    "67fc76d68dc5a9415511a4f394ef744d67510cd20e93b37cc2cc7d28e4bab65c"
   ]
  },
  "total": 3849113002
 }
}
for _k, _j in JOBS.items():
    assert sum(v[0] for v in _j["files"].values()) == _j["total"], _k
print("python", sys.version.split()[0], "| cpus", os.cpu_count())
for _k, _j in JOBS.items():
    print(f"  {_k}: {_j['hf_repo']} @ {_j['revision'][:12]} | {len(_j['files'])} files | {_j['total'] / 1e9:.2f} GB")

In [ ]:
# Kaggle Secrets -> Kaggle API credentials. Stops here (before any transfer) if they are missing or wrong.
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    KAGGLE_USERNAME = _secrets.get_secret("KAGGLE_USERNAME").strip()
    _key = _secrets.get_secret("KAGGLE_KEY").strip()
except Exception as exc:
    raise RuntimeError(
        f"Could not read Kaggle Secrets KAGGLE_USERNAME / KAGGLE_KEY ({exc!r}). Attach both (Add-ons -> Secrets) "
        "and start the run from the notebook editor (Save Version -> Save & Run All).") from None

cred_dir = Path.home() / ".kaggle"
cred_dir.mkdir(parents=True, exist_ok=True)
(cred_dir / "kaggle.json").write_text(json.dumps({"username": KAGGLE_USERNAME, "key": _key}))
(cred_dir / "kaggle.json").chmod(0o600)
os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"] = _key
del _key

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle==2.2.4"])

from kaggle.api.kaggle_api_extended import KaggleApi
from kaggle.models.upload_file import UploadFile
from kagglesdk.blobs.types.blob_api_service import ApiBlobType, ApiStartBlobUploadRequest

api = KaggleApi()
api.authenticate()
# Credential check first: kernels_list(mine=True) answers 401 for a wrong key. (Get-calls cannot be used for this:
# Kaggle answers 403 PERMISSION_DENIED both for a missing model and for someone else's private one.)
try:
    api.kernels_list(mine=True, page_size=1)
except Exception as exc:
    status = getattr(getattr(exc, "response", None), "status_code", None)
    raise RuntimeError(f"Kaggle rejected the KAGGLE_USERNAME / KAGGLE_KEY secrets (HTTP {status}). "
                       "Check the two secret values, then run again.") from None

_mine = {getattr(m, "ref", None) for m in (api.model_list(owner=KAGGLE_USERNAME, page_size=100) or [])}
for _k, _j in JOBS.items():
    _j["model_ref"] = f"{KAGGLE_USERNAME}/{_j['slug']}"
    _j["instance_ref"] = f"{_j['model_ref']}/pytorch/{_j['instance']}"
    _j["model_exists"] = _j["model_ref"] in _mine
    _j["instance_exists"] = False
    if _j["model_exists"]:
        try:
            api.model_instance_get(_j["instance_ref"])
            _j["instance_exists"] = True
        except Exception as exc:
            if getattr(getattr(exc, "response", None), "status_code", None) not in (403, 404):
                raise
    print(f"{_k}: model exists={_j['model_exists']} instance exists={_j['instance_exists']} -> {_j['instance_ref']}")
print("Kaggle API OK as", KAGGLE_USERNAME)

In [ ]:
# Stream every file Hugging Face -> Kaggle resumable upload, hashing on the fly. Nothing is written to disk.
# (Verbatim from the ARC-AGI-3 uploader; HF_REPO / HF_REVISION are set per job in the next cell.)
class _HFReader:
    """Reads one file from a byte offset in a background thread into a bounded queue (<= 48 x 8 MiB)."""

    def __init__(self, url, start, size):
        self.q = queue.Queue(maxsize=48)
        self.stop = threading.Event()
        self.error = None
        self.thread = threading.Thread(target=self._run, args=(url, start, size), daemon=True)
        self.thread.start()

    def _run(self, url, start, size):
        try:
            headers = {"Accept-Encoding": "identity"}
            if start:
                headers["Range"] = f"bytes={start}-"
            with requests.get(url, headers=headers, stream=True, timeout=(30, 300)) as r:
                r.raise_for_status()
                if start and r.status_code != 206:
                    raise RuntimeError(f"Hugging Face ignored the Range header (HTTP {r.status_code})")
                got = start
                for piece in r.iter_content(READ):
                    if self.stop.is_set():
                        return
                    got += len(piece)
                    self.q.put(piece)
                if got != size:
                    raise RuntimeError(f"Hugging Face stream ended at {got} of {size} bytes")
        except BaseException as exc:
            self.error = exc
        finally:
            self.q.put(None)

    def get(self):
        piece = self.q.get()
        if piece is None:
            raise self.error or RuntimeError("Hugging Face stream ended early")
        return piece

    def close(self):
        self.stop.set()
        try:
            while True:
                self.q.get_nowait()
        except queue.Empty:
            pass


def _start_blob(name, size):
    req = ApiStartBlobUploadRequest()
    req.type = ApiBlobType.MODEL
    req.name = name
    req.content_length = size
    req.last_modified_epoch_seconds = int(time.time())
    with api.build_kaggle_client() as kaggle:
        resp = api.with_retry(kaggle.blobs.blob_api_client.start_blob_upload)(req)
    return resp.token, resp.create_url


def _persisted(response, size):
    """Bytes the upload server has stored, from a chunk or status response."""
    if response.status_code in (200, 201):
        return size
    if response.status_code == 308:
        rng = response.headers.get("Range")
        return int(rng.rsplit("-", 1)[1]) + 1 if rng else 0
    raise RuntimeError(f"upload server answered HTTP {response.status_code}: {response.text[:300]}")


def _query(session, put_url, size):
    r = session.put(put_url, headers={"Content-Range": f"bytes */{size}", "Content-Length": "0"}, timeout=(30, 120))
    return _persisted(r, size)


def stream_file(name, size, kind, max_stalls=12):
    """Returns (token, sha256 hex, git-sha1 hex or None, seconds, failures). Gives up only after max_stalls
    consecutive failures without progress."""
    token, put_url = _start_blob(name, size)
    session = requests.Session()
    h256 = hashlib.sha256()
    gsha = hashlib.sha1(b"blob %d\0" % size) if kind == "gitsha1" else None
    hashed = persisted = 0          # hashed == bytes confirmed stored and folded into the digests
    failures = stalls = 0
    stall_mark = -1
    next_report = 5e9
    started = time.time()
    while hashed < size or persisted < size:
        reader = _HFReader(HF_URL.format(repo=HF_REPO, rev=HF_REVISION, name=name), hashed, size)
        buf = bytearray()
        buf_start = hashed
        try:
            while buf_start < size:
                while len(buf) < CHUNK and buf_start + len(buf) < size:
                    buf += reader.get()
                if persisted > buf_start:          # stored by an earlier attempt: hash only, do not resend
                    n = min(persisted - buf_start, len(buf))
                else:
                    at_end = buf_start + len(buf) == size
                    send = len(buf) if at_end else (len(buf) // QUANTUM) * QUANTUM
                    r = session.put(put_url, data=bytes(buf[:send]), timeout=(30, 900),
                                    headers={"Content-Range": f"bytes {buf_start}-{buf_start + send - 1}/{size}"})
                    persisted = _persisted(r, size)
                    n = persisted - buf_start
                    if n < 0 or n > len(buf):
                        raise RuntimeError(f"server reports {persisted} stored; chunk started at {buf_start}")
                with memoryview(buf) as mv:
                    h256.update(mv[:n])
                    if gsha is not None:
                        gsha.update(mv[:n])
                del buf[:n]
                buf_start += n
                hashed = buf_start
                if hashed >= next_report:
                    dt = time.time() - started
                    print(f"    {name}: {hashed / 1e9:.1f}/{size / 1e9:.1f} GB ({hashed / 1e6 / dt:.0f} MB/s)", flush=True)
                    next_report += 5e9
        except Exception as exc:
            failures += 1
            stalls = stalls + 1 if hashed == stall_mark else 1
            stall_mark = hashed
            print(f"    {name}: attempt failed at {hashed} bytes (stall {stalls}/{max_stalls}): {exc!r}"[:400], flush=True)
            if stalls >= max_stalls:
                raise
            session.close()
            session = requests.Session()          # drop pooled connections after a failure
            time.sleep(min(120, 10 * stalls))
            for q in range(5):
                try:
                    persisted = _query(session, put_url, size)
                    break
                except Exception as qexc:
                    if q == 4:
                        raise
                    print(f"    {name}: status query failed ({qexc!r}); retrying"[:300], flush=True)
                    time.sleep(15 * (q + 1))
            if persisted < hashed:
                raise RuntimeError(f"{name}: server now reports {persisted} bytes < {hashed} already confirmed")
        finally:
            reader.close()
    return token, h256.hexdigest(), gsha.hexdigest() if gsha is not None else None, time.time() - started, failures


def upload_bytes(name, data):
    token, put_url = _start_blob(name, len(data))
    r = requests.put(put_url, data=data, headers={"Content-Range": f"bytes 0-{len(data) - 1}/{len(data)}"},
                     timeout=(30, 300))
    if _persisted(r, len(data)) != len(data):
        raise RuntimeError(f"{name}: small upload incomplete")
    return token

In [ ]:
# Per job: stream + verify every file, then create the model (if new) and the instance with the streamed blob tokens.
RESULTS = {}
for JOB, J in JOBS.items():
    print(f"\n==================== {JOB}: {J['hf_repo']} -> {J['instance_ref']}")
    if J["instance_exists"]:
        print("instance already exists: transfer skipped")
        RESULTS[JOB] = "exists"
        continue
    HF_REPO, HF_REVISION = J["hf_repo"], J["revision"]
    EXPECTED = {n: tuple(v) for n, v in J["files"].items()}
    todo = sorted((n for n in EXPECTED if n not in NOT_UPLOADED), key=lambda n: -EXPECTED[n][0])
    TOKENS, HASHES = {}, {}
    t0 = time.time()

    def _one(name):
        size, kind, want = EXPECTED[name]
        print(f"[{time.strftime('%H:%M:%S')}] start {name} ({size / 1e9:.2f} GB)", flush=True)
        for attempt in range(1, 4):      # a fresh upload session per attempt (e.g. if the old one expired)
            try:
                token, s256, g1, dt, fails = stream_file(name, size, kind)
                break
            except Exception as exc:
                print(f"[{time.strftime('%H:%M:%S')}] {name}: file attempt {attempt}/3 failed: {exc!r}"[:400], flush=True)
                if attempt == 3:
                    raise
        got = s256 if kind == "sha256" else g1
        print(f"[{time.strftime('%H:%M:%S')}] done  {name} in {dt:.0f} s ({size / 1e6 / max(dt, 1e-6):.0f} MB/s, "
              f"{fails} retries) {kind} {'OK' if got == want else 'MISMATCH'}", flush=True)
        return name, token, s256, got == want

    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        for name, token, s256, ok in pool.map(_one, todo):
            TOKENS[name], HASHES[name] = token, (s256, ok)
    bad = sorted(n for n, (_, ok) in HASHES.items() if not ok)
    total = sum(EXPECTED[n][0] for n in todo)
    dt = time.time() - t0
    print(f"streamed {len(todo)} files, {total / 1e9:.2f} GB in {dt / 60:.1f} min ({total / 1e6 / max(dt, 1e-6):.0f} MB/s)")
    if bad:
        raise RuntimeError(f"{JOB}: hash mismatch, instance NOT created: {bad}")
    print(f"all {len(todo)} files match the pinned Hugging Face revision byte-for-byte")
    manifest = {
        "source": {"hf_repo": HF_REPO, "hf_revision": HF_REVISION},
        "verified_against": "Hugging Face tree API at the pinned revision (LFS sha256 / git blob sha1), hashed while streaming",
        "files": {n: {"bytes": EXPECTED[n][0], "sha256": HASHES[n][0]} for n in sorted(todo)},
        "total_bytes": total, "not_uploaded": sorted(NOT_UPLOADED & set(EXPECTED)),
        "built_for": "ARC-AGI-2 Experiment 11: SGLang 0.5.21 + RedHat Qwen3.8-27B INT4 + DFlash2 on 4x L4",
    }
    TOKENS["KAGGLE_MIRROR_MANIFEST.json"] = upload_bytes(
        "KAGGLE_MIRROR_MANIFEST.json", (json.dumps(manifest, indent=1) + "\n").encode("utf-8"))

    if not J["model_exists"]:
        parent_dir = Path(f"/tmp/kaggle-model-parent-{JOB}")
        shutil.rmtree(parent_dir, ignore_errors=True)
        parent_dir.mkdir(parents=True)
        (parent_dir / "model-metadata.json").write_text(json.dumps({
            "ownerSlug": KAGGLE_USERNAME, "title": J["title"], "slug": J["slug"], "subtitle": J["subtitle"],
            "isPrivate": not PUBLIC,
            "description": (f"Byte-identical mirror of {HF_REPO} (revision {HF_REVISION}) for offline Kaggle inference. "
                            "License: Apache-2.0 (see README.md in the files). Hashes in KAGGLE_MIRROR_MANIFEST.json."),
            "licenseName": LICENSE_NAME,
        }, indent=2))
        created = api.model_create_new(str(parent_dir))
        if getattr(created, "error", None):
            raise RuntimeError(f"{JOB}: model create failed: {created.error}")
        print("model created:", getattr(created, "ref", None), getattr(created, "url", None))

    # model_instance_create walks a folder and asks _upload_file for a token per file. The folder holds empty
    # placeholders with the real names; the patched _upload_file returns the token of the streamed blob.
    ph = Path(f"/tmp/kaggle-instance-placeholders-{JOB}")
    shutil.rmtree(ph, ignore_errors=True)
    ph.mkdir(parents=True)
    for name in TOKENS:
        (ph / name).touch()
    (ph / "model-instance-metadata.json").write_text(json.dumps({
        "ownerSlug": KAGGLE_USERNAME, "modelSlug": J["slug"], "instanceSlug": J["instance"],
        "framework": KAGGLE_FRAMEWORK,
        "overview": f"{HF_REPO} at {HF_REVISION}, uploaded byte-identical (hashes in KAGGLE_MIRROR_MANIFEST.json).",
        "usage": "Mount ${PATH}; serve offline with SGLang 0.5.21 (see the Experiment 11 notebooks).",
        "licenseName": LICENSE_NAME, "fineTunable": False, "trainingData": [], "modelInstanceType": "Unspecified",
        "baseModelInstance": "",
        "externalBaseModelUrl": "",
    }, indent=2))
    used = []
    _original_upload_file = KaggleApi._upload_file

    def _token_for(self, file_name, full_path, blob_type, upload_context, quiet, resources, content_type=None):
        if file_name not in TOKENS:
            raise RuntimeError(f"no streamed blob for {file_name}")
        used.append(file_name)
        return UploadFile(token=TOKENS[file_name])

    KaggleApi._upload_file = _token_for
    try:
        response = api.model_instance_create(str(ph), quiet=True, dir_mode="skip")
    finally:
        KaggleApi._upload_file = _original_upload_file
    if sorted(used) != sorted(TOKENS):
        raise RuntimeError(f"{JOB}: token hand-off mismatch: used={sorted(used)}")
    if getattr(response, "error", None):
        raise RuntimeError(f"{JOB}: instance create failed: {response.error}")
    print(f"instance created with {len(used)} files:", getattr(response, "ref", None), getattr(response, "url", None))
    RESULTS[JOB] = "created"
print("\nresults:", RESULTS)

In [ ]:
# List what Kaggle now holds (processing can lag a few minutes behind the create call).
for JOB, J in JOBS.items():
    expected = {n: J["files"][n][0] for n in J["files"] if n not in NOT_UPLOADED}
    listed = {}
    for attempt in range(20):
        try:
            resp = api.model_instance_version_files(f"{J['instance_ref']}/1", page_size=100)
            listed = {f.name: f.size for f in (getattr(resp, "files", None) or [])}
        except Exception as exc:
            print("listing not ready:", repr(exc)[:200])
        if set(expected) <= set(listed):
            break
        time.sleep(60)
    print(f"\n{JOB}: {len(listed)} files listed for {J['instance_ref']}/1")
    for name in sorted(listed):
        print(f"  {listed[name]:>14,}  {name}")
    missing = sorted(set(expected) - set(listed))
    wrong = {n: (listed[n], expected[n]) for n in expected if n in listed and listed[n] != expected[n]}
    print("missing:", missing or "none", "| size mismatches:", wrong or "none")
    print("attach with:", f'"model_sources": ["{J["instance_ref"]}/1"]', "| mount:", f"/kaggle/input/models/{J['instance_ref']}/1")